# 🏠 Household Load Profiles & PV Analysis

Analyse household electricity consumption patterns, photovoltaic generation, and self-consumption optimization.

**Sensors used:**
- `Household power draw` — total household electricity (W), 435K readings
- `Photovoltaics Power` — PV generation (W), 93K readings
- `warmepumpe_Energie_sum` — heat pump cumulative energy (kWh)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from app.database.database import get_db_session
from app.database.models import Sensor, SensorReading

print("✅ Imports ready")

✅ Imports ready


## 1. Load Household & PV Data

Query power readings for household consumption and photovoltaic generation, resample to hourly averages, and convert W → kWh.

In [3]:
def load_power_series(db, sensor_name):
    """Load sensor readings and return hourly kWh series."""
    sensor = db.query(Sensor).filter(Sensor.name == sensor_name).first()
    rows = (db.query(SensorReading.timestamp, SensorReading.value)
            .filter(SensorReading.sensor_id == sensor.id)
            .order_by(SensorReading.timestamp).all())
    s = pd.DataFrame(rows, columns=['timestamp', 'watts'])
    s['timestamp'] = pd.to_datetime(s['timestamp'], utc=True)
    s = s.set_index('timestamp')
    hourly = s.resample('1h').mean()
    hourly['kwh'] = hourly['watts'] / 1000  # W avg over 1h ≈ kWh
    return hourly[['kwh']].dropna()

with get_db_session() as db:
    household = load_power_series(db, 'Household power draw').rename(columns={'kwh': 'household_kwh'})
    pv = load_power_series(db, 'Photovoltaics Power').rename(columns={'kwh': 'pv_kwh'})

df = household.join(pv, how='outer').fillna(0)
df['net_kwh'] = df['household_kwh'] - df['pv_kwh']  # positive = grid draw
df['self_consumed'] = df[['household_kwh', 'pv_kwh']].min(axis=1)
df['grid_export'] = (df['pv_kwh'] - df['household_kwh']).clip(lower=0)
df['grid_import'] = (df['household_kwh'] - df['pv_kwh']).clip(lower=0)

print(f"✅ Household: {len(household)} hours | PV: {len(pv)} hours | Combined: {len(df)} hours")
print(f"   Date range: {df.index.min().date()} → {df.index.max().date()}")
print(f"   Total consumption: {df['household_kwh'].sum():.0f} kWh")
print(f"   Total PV generation: {df['pv_kwh'].sum():.0f} kWh")
print(f"   Self-consumption: {df['self_consumed'].sum():.0f} kWh ({100*df['self_consumed'].sum()/df['pv_kwh'].sum():.1f}%)")

2026-03-24 15:22:25,885 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-24 15:22:25,886 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:22:25,904 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-24 15:22:25,905 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:22:25,922 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-24 15:22:25,923 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:22:25,940 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:22:25,943 INFO sqlalchemy.engine.Engine SELECT api_sensors.id AS api_sensors_id, api_sensors.device_id AS api_sensors_device_id, api_sensors.name AS api_sensors_name, api_sensors.description AS api_sensors_description, api_sensors.sensor_type_id AS api_sensors_sensor_type_id, api_sensors.location_id AS api_sensors_location_id, api_sensors.manufacturer AS api_sensors_manufacturer, api_sensors.model AS api_sensors_model, api_sensors.firmware_version AS api_sensors_

## 2. Daily Energy Balance

Stacked area chart showing daily grid import, self-consumption, and grid export to visualize energy flow patterns.

In [4]:
daily = df.resample('1D').sum()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily.index, y=daily['grid_import'], mode='lines', name='Grid Import',
                         fill='tozeroy', line=dict(color='#EF553B')))
fig.add_trace(go.Scatter(x=daily.index, y=daily['self_consumed'], mode='lines', name='Self-Consumed PV',
                         fill='tozeroy', line=dict(color='#00CC96')))
fig.add_trace(go.Scatter(x=daily.index, y=-daily['grid_export'], mode='lines', name='Grid Export',
                         fill='tozeroy', line=dict(color='#636EFA')))
fig.update_layout(
    title='Daily Energy Balance',
    yaxis_title='Energy (kWh)', xaxis_title='',
    height=450, legend=dict(x=0.01, y=0.99)
)
fig.show()

## 3. Average Load Profile by Hour

Mean hourly consumption and PV generation across different day types (weekday vs weekend), showing typical daily curves.

In [ ]:
df['hour'] = df.index.hour
df['is_weekend'] = df.index.dayofweek >= 5
df['day_type'] = df['is_weekend'].map({True: 'Weekend', False: 'Weekday'})

profile = df.groupby(['day_type', 'hour'])[['household_kwh', 'pv_kwh']].mean().reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Weekday', 'Weekend'], shared_yaxes=True)
for i, dtype in enumerate(['Weekday', 'Weekend'], 1):
    sub = profile[profile['day_type'] == dtype]
    fig.add_trace(go.Scatter(x=sub['hour'], y=sub['household_kwh'], mode='lines+markers',
                             name='Consumption', line=dict(color='#EF553B'),
                             showlegend=(i == 1)), row=1, col=i)
    fig.add_trace(go.Scatter(x=sub['hour'], y=sub['pv_kwh'], mode='lines+markers',
                             name='PV Generation', line=dict(color='#00CC96'),
                             showlegend=(i == 1)), row=1, col=i)
    fig.add_trace(go.Scatter(x=sub['hour'], y=sub['pv_kwh'], fill='tonexty',
                             fillcolor='rgba(0,204,150,0.15)', showlegend=False,
                             line=dict(width=0)), row=1, col=i)
    fig.update_xaxes(title_text='Hour', row=1, col=i)

fig.update_yaxes(title_text='Average Power (kWh)', row=1, col=1)
fig.update_layout(title='Average Hourly Load Profile', height=400)
fig.show()

## 4. Peak Load Analysis

Identify the top consumption hours and visualize the distribution of hourly consumption to understand peak demand patterns.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Consumption Distribution', 'Top 20 Peak Hours'])

# Histogram of hourly consumption
fig.add_trace(go.Histogram(x=df['household_kwh'], nbinsx=100, marker_color='#636EFA',
                           name='Hours'), row=1, col=1)
fig.update_xaxes(title_text='Hourly Consumption (kWh)', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)

# Top 20 peak hours
peaks = df.nlargest(20, 'household_kwh')[['household_kwh']].copy()
peaks['label'] = peaks.index.strftime('%b %d %H:%M')
fig.add_trace(go.Bar(x=peaks['label'], y=peaks['household_kwh'],
                     marker_color='#EF553B', name='Peak'), row=1, col=2)
fig.update_xaxes(title_text='', tickangle=45, row=1, col=2)
fig.update_yaxes(title_text='Consumption (kWh)', row=1, col=2)

p95 = df['household_kwh'].quantile(0.95)
p99 = df['household_kwh'].quantile(0.99)
fig.update_layout(title=f'Peak Load Analysis — P95: {p95:.2f} kWh, P99: {p99:.2f} kWh',
                  height=450, showlegend=False)
fig.show()

## 5. Weekly Heatmap — Consumption by Hour & Day

Reveals recurring consumption patterns across the week using a heatmap of average hourly consumption.

In [ ]:
df['dow'] = df.index.dayofweek
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

heatmap_data = df.pivot_table(values='household_kwh', index='hour', columns='dow', aggfunc='mean')
heatmap_data.columns = day_names

fig = go.Figure(go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='YlOrRd',
    colorbar_title='Avg kWh'
))
fig.update_layout(
    title='Average Hourly Consumption by Day of Week',
    xaxis_title='Day', yaxis_title='Hour of Day',
    yaxis=dict(dtick=1), height=500
)
fig.show()

## 6. Monthly Energy Summary & Self-Sufficiency

Monthly totals for consumption, PV generation, self-consumption, grid import/export with self-sufficiency ratio.

In [ ]:
monthly = df.resample('ME')[['household_kwh', 'pv_kwh', 'self_consumed', 'grid_import', 'grid_export']].sum()
monthly['self_sufficiency'] = monthly['self_consumed'] / monthly['household_kwh'] * 100
monthly['month_label'] = monthly.index.strftime('%b %Y')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=['Monthly Energy (kWh)', 'Self-Sufficiency Rate (%)'],
                    row_heights=[0.65, 0.35])

fig.add_trace(go.Bar(x=monthly['month_label'], y=monthly['household_kwh'],
                     name='Consumption', marker_color='#EF553B'), row=1, col=1)
fig.add_trace(go.Bar(x=monthly['month_label'], y=monthly['pv_kwh'],
                     name='PV Generation', marker_color='#00CC96'), row=1, col=1)
fig.add_trace(go.Bar(x=monthly['month_label'], y=monthly['self_consumed'],
                     name='Self-Consumed', marker_color='#FFA15A'), row=1, col=1)

fig.add_trace(go.Scatter(x=monthly['month_label'], y=monthly['self_sufficiency'],
                         mode='lines+markers+text', text=monthly['self_sufficiency'].round(1).astype(str) + '%',
                         textposition='top center', line=dict(color='#AB63FA', width=3),
                         name='Self-Sufficiency'), row=2, col=1)
fig.add_hline(y=50, line_dash='dash', line_color='gray', annotation_text='50%', row=2, col=1)

fig.update_layout(title='Monthly Energy Summary', height=600, barmode='group')
fig.update_yaxes(title_text='kWh', row=1, col=1)
fig.update_yaxes(title_text='%', range=[0, 100], row=2, col=1)
fig.show()

monthly[['household_kwh', 'pv_kwh', 'self_consumed', 'grid_import', 'grid_export', 'self_sufficiency']].round(1)

## 7. Typical Day Overlay — Seasonal Comparison

Compare the average daily load profile across seasons (summer vs winter) to highlight seasonal shifts in consumption and PV generation.

In [ ]:
df['month'] = df.index.month
season_map = {12: 'Winter', 1: 'Winter', 2: 'Winter',
              3: 'Spring', 4: 'Spring', 5: 'Spring',
              6: 'Summer', 7: 'Summer', 8: 'Summer',
              9: 'Autumn', 10: 'Autumn', 11: 'Autumn'}
df['season'] = df['month'].map(season_map)

colors = {'Winter': '#636EFA', 'Spring': '#00CC96', 'Summer': '#FFA15A', 'Autumn': '#AB63FA'}

fig = make_subplots(rows=1, cols=2, subplot_titles=['Household Consumption', 'PV Generation'], shared_yaxes=True)

for season in ['Winter', 'Spring', 'Summer', 'Autumn']:
    sub = df[df['season'] == season]
    if len(sub) == 0:
        continue
    hourly_avg = sub.groupby('hour')['household_kwh'].mean()
    pv_avg = sub.groupby('hour')['pv_kwh'].mean()
    fig.add_trace(go.Scatter(x=hourly_avg.index, y=hourly_avg.values, mode='lines',
                             name=season, line=dict(color=colors[season], width=2),
                             showlegend=True), row=1, col=1)
    fig.add_trace(go.Scatter(x=pv_avg.index, y=pv_avg.values, mode='lines',
                             name=season, line=dict(color=colors[season], width=2, dash='dot'),
                             showlegend=False), row=1, col=2)

fig.update_xaxes(title_text='Hour', dtick=2)
fig.update_yaxes(title_text='Avg kWh', row=1, col=1)
fig.update_layout(title='Seasonal Load Profiles', height=400)
fig.show()